⚠️ **Security note:** This notebook uses `dbutils.secrets` to retrieve the ADLS storage key.
Never hardcode credentials. The secret scope `adls` must be configured before running:
```
databricks secrets create-scope --scope adls
databricks secrets put-secret --scope adls --key storage-account-key
```

# Silver Layer — Road Network ETL Pipeline
Reads Bronze parquets from ADLS, applies cleaning and transformations, writes Silver Delta tables back to ADLS.

**Inputs (Bronze):**
- `bronze/osm/christchurch/2026-03-20/road_segments.parquet`
- `bronze/osm/christchurch/2026-03-20/road_nodes.parquet`
- `bronze/waka_kotahi/2026-03-20/tms_daily_traffic_counts.parquet`
- `bronze/waka_kotahi/2026-03-20/tms_monitoring_sites.parquet`

**Outputs (Silver Delta tables):**
- `silver/osm/road_segments_silver`
- `silver/waka_kotahi/tms_counts_silver`
- `silver/waka_kotahi/tms_monitoring_sites_silver`

In [0]:
# ── ADLS Configuration ───────────────────────────────────
STORAGE_ACCOUNT = "nzetlpipeline"
CONTAINER       = "medallion"
KEY             = dbutils.secrets.get(scope="adls", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    KEY
)

BASE_PATH   = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# ── Date parameter — passed from Databricks job or defaults to today ─────────
dbutils.widgets.text("bronze_date", "2026-03-20", "Bronze Date (YYYY-MM-DD)")
BRONZE_DATE = dbutils.widgets.get("bronze_date")

# Bronze input paths
BRONZE_OSM  = f"{BASE_PATH}/bronze/osm/christchurch/{BRONZE_DATE}"
BRONZE_WK   = f"{BASE_PATH}/bronze/waka_kotahi/{BRONZE_DATE}"

# Silver output paths (Delta)
SILVER_SEGMENTS = f"{BASE_PATH}/silver/osm/road_segments_silver"
SILVER_COUNTS   = f"{BASE_PATH}/silver/waka_kotahi/tms_counts_silver"
SILVER_SITES    = f"{BASE_PATH}/silver/waka_kotahi/tms_monitoring_sites_silver"

print(f"Config loaded. Bronze date: {BRONZE_DATE}")
print(f"Bronze OSM  : {BRONZE_OSM}")
print(f"Bronze WK   : {BRONZE_WK}")

Config loaded. Bronze date: 2026-03-20
Bronze OSM  : abfss://medallion@nzetlpipeline.dfs.core.windows.net/bronze/osm/christchurch/2026-03-20
Bronze WK   : abfss://medallion@nzetlpipeline.dfs.core.windows.net/bronze/waka_kotahi/2026-03-20


## 1. Verify Bronze inputs are accessible

In [0]:
for path in [BRONZE_OSM, BRONZE_WK]:
    files = dbutils.fs.ls(path)
    print(f"\n{path}")
    for f in files:
        size_kb = f.size / 1024
        print(f"  {f.name}  ({size_kb:.1f} KB)")


abfss://medallion@nzetlpipeline.dfs.core.windows.net/bronze/osm/christchurch/2026-03-20
  manifest.json  (0.8 KB)
  road_nodes.parquet  (484.8 KB)
  road_segments.parquet  (2895.3 KB)

abfss://medallion@nzetlpipeline.dfs.core.windows.net/bronze/waka_kotahi/2026-03-20
  manifest.json  (0.5 KB)
  tms_daily_traffic_counts.parquet  (34919.1 KB)
  tms_monitoring_sites.parquet  (177.6 KB)


## 2. Silver — OSM Road Segments
Cleans maxspeed, classifies highway type, reprojects geometry to EPSG:2193 (NZTM2000).

Note: Geometry reprojection is handled via pandas/geopandas UDF since PySpark
does not natively support CRS transformations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType
import pandas as pd
import geopandas as gpd
from shapely import wkb, wkt
from pyproj import Transformer

# ── Read Bronze ──────────────────────────────────────────
segments_bronze = spark.read.parquet(f"{BRONZE_OSM}/road_segments.parquet")
print(f"Bronze segments: {segments_bronze.count():,} rows")
segments_bronze.printSchema()

Bronze segments: 25,366 rows
root
 |-- u: long (nullable = true)
 |-- v: long (nullable = true)
 |-- key: long (nullable = true)
 |-- osmid: string (nullable = true)
 |-- highway: string (nullable = true)
 |-- lanes: string (nullable = true)
 |-- name: string (nullable = true)
 |-- oneway: boolean (nullable = true)
 |-- reversed: string (nullable = true)
 |-- length: double (nullable = true)
 |-- geometry: binary (nullable = true)
 |-- junction: string (nullable = true)
 |-- maxspeed: string (nullable = true)
 |-- ref: string (nullable = true)
 |-- bridge: string (nullable = true)
 |-- access: string (nullable = true)
 |-- width: string (nullable = true)
 |-- tunnel: string (nullable = true)



In [0]:
# ── Clean maxspeed ───────────────────────────────────────
# maxspeed arrives as string or list — extract integer value

def clean_maxspeed(val):
    """Extract integer speed from maxspeed string/list. Returns None if unparseable."""
    if val is None:
        return None
    # If it's a list-like string e.g. "['50', '60']" take the first element
    if val.startswith("["):
        try:
            import ast
            val = ast.literal_eval(val)
            val = val[0] if isinstance(val, list) and val else None
        except Exception:
            return None
    if val is None:
        return None
    # Strip non-numeric suffixes e.g. "50 mph", "50 km/h"
    val = str(val).strip().split()[0]
    try:
        speed = int(float(val))
        # Convert mph to km/h if needed (NZ uses km/h so this is a safety check)
        # Typical NZ speeds: 10, 20, 30, 40, 50, 60, 80, 100, 110
        return speed if 5 <= speed <= 150 else None
    except (ValueError, TypeError):
        return None

clean_maxspeed_udf = F.udf(clean_maxspeed, IntegerType())

# ── Classify highway ─────────────────────────────────────
# Map OSM highway tags to simplified road classes

def classify_highway(val):
    if val is None:
        return "unclassified"
    motorway   = {"motorway", "motorway_link"}
    arterial   = {"trunk", "trunk_link", "primary", "primary_link",
                  "secondary", "secondary_link"}
    collector  = {"tertiary", "tertiary_link", "unclassified"}
    local      = {"residential", "living_street", "service"}
    active     = {"cycleway", "footway", "path", "pedestrian", "steps"}
    # Handle list-like strings
    if val.startswith("["):
        try:
            import ast
            val = ast.literal_eval(val)
            val = val[0] if isinstance(val, list) and val else ""
        except Exception:
            return "other"
    val = str(val).strip()
    if val in motorway:  return "motorway"
    if val in arterial:  return "arterial"
    if val in collector: return "collector"
    if val in local:     return "local"
    if val in active:    return "active_transport"
    return "other"

classify_highway_udf = F.udf(classify_highway, StringType())

In [0]:
# ── Apply transformations ────────────────────────────────
segments_silver = (
    segments_bronze
    .withColumn("maxspeed_kmh",  clean_maxspeed_udf(F.col("maxspeed")))
    .withColumn("road_class",    classify_highway_udf(F.col("highway")))
    .withColumn("oneway",        F.col("oneway").cast("boolean"))
    .withColumn("length",        F.col("length").cast(DoubleType()))
    .drop("maxspeed")   # replaced by maxspeed_kmh
)

print(f"Silver segments: {segments_silver.count():,} rows")
segments_silver.printSchema()
segments_silver.select(
    "osmid", "name", "road_class", "maxspeed_kmh", "oneway", "length"
).show(5, truncate=False)

Silver segments: 25,366 rows
root
 |-- u: long (nullable = true)
 |-- v: long (nullable = true)
 |-- key: long (nullable = true)
 |-- osmid: string (nullable = true)
 |-- highway: string (nullable = true)
 |-- lanes: string (nullable = true)
 |-- name: string (nullable = true)
 |-- oneway: boolean (nullable = true)
 |-- reversed: string (nullable = true)
 |-- length: double (nullable = true)
 |-- geometry: binary (nullable = true)
 |-- junction: string (nullable = true)
 |-- ref: string (nullable = true)
 |-- bridge: string (nullable = true)
 |-- access: string (nullable = true)
 |-- width: string (nullable = true)
 |-- tunnel: string (nullable = true)
 |-- maxspeed_kmh: integer (nullable = true)
 |-- road_class: string (nullable = true)

+--------------------------------+------------------------+----------+------------+------+------------------+
|osmid                           |name                    |road_class|maxspeed_kmh|oneway|length            |
+------------------------------

### Geometry reprojection — WGS84 → NZTM2000 (EPSG:2193)
We use a pandas UDF (vectorised) to reproject geometry via geopandas.
The geometry column is stored as WKB bytes in parquet.

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import BinaryType
import pandas as pd
import geopandas as gpd

@pandas_udf(BinaryType())
def reproject_geometry(geom_series: pd.Series) -> pd.Series:
    """Reproject WKB geometry from EPSG:4326 to EPSG:2193 (NZTM2000)."""
    # Parse WKB to GeoSeries
    geom = gpd.GeoSeries.from_wkb(geom_series, crs="EPSG:4326")
    # Reproject
    geom_nztm = geom.to_crs("EPSG:2193")
    # Return as WKB bytes
    return geom_nztm.to_wkb()

segments_silver = segments_silver.withColumn(
    "geometry", reproject_geometry(F.col("geometry"))
)

print("Geometry reprojected to EPSG:2193.")

Geometry reprojected to EPSG:2193.


In [0]:
# ── Write Silver Delta table ─────────────────────────────
(
    segments_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_SEGMENTS)
)
print(f"Saved: {SILVER_SEGMENTS}")

# Quick read-back check
spark.read.format("delta").load(SILVER_SEGMENTS).select(
    "osmid", "road_class", "maxspeed_kmh", "length"
).show(5)

Saved: abfss://medallion@nzetlpipeline.dfs.core.windows.net/silver/osm/road_segments_silver
+--------------------+----------+------------+------------------+
|               osmid|road_class|maxspeed_kmh|            length|
+--------------------+----------+------------+------------------+
|[1133615064, 5365...| collector|        NULL|56.928545045026226|
|           488963306|  arterial|          50| 29.95627054336464|
|             4243735|  arterial|          50| 4.084168735002902|
|          1035329059|  arterial|          50|4.7344343679711836|
|           837770558|  arterial|          50| 6.123828064908569|
+--------------------+----------+------------+------------------+
only showing top 5 rows


## 3. Silver — TMS Traffic Counts
Canterbury only, dates parsed, trafficCount → Int64,
aggregated to site/date/vehicle_class grain.

In [0]:
counts_bronze = spark.read.parquet(f"{BRONZE_WK}/tms_daily_traffic_counts.parquet")
print(f"Bronze counts: {counts_bronze.count():,} rows")
counts_bronze.printSchema()

Bronze counts: 4,640,033 rows
root
 |-- startDate: string (nullable = true)
 |-- siteID: long (nullable = true)
 |-- regionName: string (nullable = true)
 |-- siteReference: string (nullable = true)
 |-- classWeight: string (nullable = true)
 |-- siteDescription: string (nullable = true)
 |-- laneNumber: long (nullable = true)
 |-- flowDirection: long (nullable = true)
 |-- trafficCount: double (nullable = true)



In [0]:

# Check what region values actually exist
counts_bronze.groupBy("regionName").count().orderBy("count", ascending=False).show()

+--------------------+-------+
|          regionName|  count|
+--------------------+-------+
|       02 - Auckland|1315767|
|     11 - Canterbury| 765697|
|     09 - Wellington| 585558|
|          13 - Otago| 409314|
|10 - Nelson/Marlb...| 215243|
|08 - Manawatu-Wan...| 204786|
|      01 - Northland| 175166|
|        03 - Waikato| 173964|
|     12 - West Coast| 170579|
|  04 - Bay of Plenty| 164057|
|     06 - Hawkes Bay| 157150|
|      14 - Southland| 142884|
|       07 - Taranaki|  95926|
|       05 - Gisborne|  63942|
+--------------------+-------+



In [0]:
from pyspark.sql.types import LongType, DateType

counts_silver = (
    counts_bronze
    # Canterbury only
    .filter(F.col("regionName") == "11 - Canterbury")
    # Parse date
    .withColumn("date", F.to_date(F.col("startDate"), "dd/MM/yyyy"))
    .withColumn("year",  F.year("date").cast("short"))
    .withColumn("month", F.month("date").cast("byte"))
    # Cast count to long (nullable)
    .withColumn("daily_count", F.col("trafficCount").cast(LongType()))
    # Rename vehicle class
    .withColumnRenamed("classWeight", "vehicle_class")
    # Aggregate to site/date/vehicle_class grain
    # (handles duplicate lane entries — sum across lanes)
    .groupBy("date", "year", "month", "siteID", "siteReference",
             "regionName", "vehicle_class")
    .agg(F.sum("daily_count").alias("daily_count"))
    # Drop original date string
    .drop("startDate")
)

print(f"Silver counts: {counts_silver.count():,} rows")
counts_silver.printSchema()
counts_silver.show(5)

Silver counts: 389,791 rows
root
 |-- date: date (nullable = true)
 |-- year: short (nullable = true)
 |-- month: byte (nullable = true)
 |-- siteID: long (nullable = true)
 |-- siteReference: string (nullable = true)
 |-- regionName: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- daily_count: long (nullable = true)

+----------+----+-----+------+-------------+---------------+-------------+-----------+
|      date|year|month|siteID|siteReference|     regionName|vehicle_class|daily_count|
+----------+----+-----+------+-------------+---------------+-------------+-----------+
|2019-01-01|2019|    1|478055|     01S20337|11 - Canterbury|        Light|       8107|
|2020-01-01|2020|    1|478050|     01S10332|11 - Canterbury|        Light|       6759|
|2022-01-01|2022|    1|   544|     01S00389|11 - Canterbury|        Light|      10354|
|2022-01-01|2022|    1|  7981|     07620013|11 - Canterbury|        Light|      10328|
|2023-01-01|2023|    1|  3042|     07310004|

In [0]:
(
    counts_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_COUNTS)
)
print(f"Saved: {SILVER_COUNTS}")

Saved: abfss://medallion@nzetlpipeline.dfs.core.windows.net/silver/waka_kotahi/tms_counts_silver


## 4. Silver — TMS Monitoring Sites
Reprojects coordinates from NZTM2000 (already EPSG:2193),
drops obsolete/failed sites.

In [0]:
sites_bronze = spark.read.parquet(f"{BRONZE_WK}/tms_monitoring_sites.parquet")
print(f"Bronze sites: {sites_bronze.count():,} rows")
sites_bronze.printSchema()

Bronze sites: 2,042 rows
root
 |-- X: double (nullable = true)
 |-- Y: double (nullable = true)
 |-- region: string (nullable = true)
 |-- sh: string (nullable = true)
 |-- rs: long (nullable = true)
 |-- rp: double (nullable = true)
 |-- description: string (nullable = true)
 |-- lane: string (nullable = true)
 |-- type: string (nullable = true)
 |-- equipmentcurrent: string (nullable = true)
 |-- accepteddays: double (nullable = true)
 |-- siteref: string (nullable = true)
 |-- sitetype: string (nullable = true)
 |-- percentheavy: double (nullable = true)
 |-- aadt5yearsago: double (nullable = true)
 |-- aadt4yearsago: double (nullable = true)
 |-- aadt3yearsago: double (nullable = true)
 |-- aadt2yearsago: double (nullable = true)
 |-- aadt1yearago: double (nullable = true)
 |-- OBJECTID: long (nullable = true)



In [0]:
# Get the set of siteReferences that appear in counts
active_site_refs = counts_silver.select("siteReference").distinct()

# Filter sites to only Canterbury sites with actual counts
sites_silver = (
    sites_bronze
    .withColumnRenamed("X", "easting")
    .withColumnRenamed("Y", "northing")
    .withColumnRenamed("siteref", "siteReference")
    .withColumnRenamed("description", "site_description")
    .filter(F.col("easting").isNotNull() & F.col("northing").isNotNull())
    .filter(~F.col("site_description").rlike("(?i)obsolete|decommission"))
    # ← Add this join to keep only sites that have counts
    .join(active_site_refs, on="siteReference", how="inner")
    .select(
        "siteReference", "region", "site_description",
        "easting", "northing", "lane", "type", "sitetype",
        "percentheavy", "aadt1yearago", "aadt2yearsago",
        "aadt3yearsago", "aadt4yearsago", "aadt5yearsago",
    )
)

In [0]:
# sites_silver = (
#     sites_bronze
#     .withColumnRenamed("X", "easting")
#     .withColumnRenamed("Y", "northing")
#     .withColumnRenamed("siteref", "siteReference")
#     .withColumnRenamed("description", "site_description")
#     # Drop rows with null coordinates
#     .filter(F.col("easting").isNotNull() & F.col("northing").isNotNull())
#     # Drop obsolete/decommissioned sites
#     .filter(~F.col("site_description").rlike("(?i)obsolete|decommission"))
#     .select(
#         "siteReference",
#         "region",
#         "site_description",
#         "easting",
#         "northing",
#         "lane",
#         "type",
#         "sitetype",
#         "percentheavy",
#         "aadt1yearago",
#         "aadt2yearsago",
#         "aadt3yearsago",
#         "aadt4yearsago",
#         "aadt5yearsago",
#     )
# )

print(f"Silver sites: {sites_silver.count():,} rows")
sites_silver.show(5, truncate=False)

Silver sites: 233 rows
+-------------+---------------+-----------------------------------------------+------------+------------+----+--------------+-----------------------+------------+------------+-------------+-------------+-------------+-------------+
|siteReference|region         |site_description                               |easting     |northing    |lane|type          |sitetype               |percentheavy|aadt1yearago|aadt2yearsago|aadt3yearsago|aadt4yearsago|aadt5yearsago|
+-------------+---------------+-----------------------------------------------+------------+------------+----+--------------+-----------------------+------------+------------+-------------+-------------+-------------+-------------+
|08200069     |11 - Canterbury|Kurow at Waitaki River bridge No 1             |1400351.8569|5044257.0216|Both|Continuous    |Regional Continuous    |2.3         |1021.0      |996.0        |956.0        |943.0        |901.0        |
|07410008     |11 - Canterbury|CNC - Hills/QEII S

In [0]:
(
    sites_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_SITES)
)
print(f"Saved: {SILVER_SITES}")

Saved: abfss://medallion@nzetlpipeline.dfs.core.windows.net/silver/waka_kotahi/tms_monitoring_sites_silver


## 5. Validation

In [0]:
checks = {
    "road_segments_silver" : spark.read.format("delta").load(SILVER_SEGMENTS),
    "tms_counts_silver"    : spark.read.format("delta").load(SILVER_COUNTS),
    "tms_sites_silver"     : spark.read.format("delta").load(SILVER_SITES),
}

print(f"{'Table':<30} {'Rows':>10} {'Cols':>6}")
print("-" * 50)
for name, df in checks.items():
    print(f"{name:<30} {df.count():>10,} {len(df.columns):>6}")

# Null check on key measure
null_counts = spark.read.format("delta").load(SILVER_COUNTS)
null_n = null_counts.filter(F.col("daily_count").isNull()).count()
total  = null_counts.count()
print(f"\ndaily_count nulls: {null_n:,} / {total:,} ({(null_n/total*100) if total != 0 else 0:.2f}%)")

# Date range
null_counts.agg(
    F.min("date").alias("earliest"),
    F.max("date").alias("latest")
).show()

Table                                Rows   Cols
--------------------------------------------------
road_segments_silver               25,366     19
tms_counts_silver                 389,791      8
tms_sites_silver                      233     14

daily_count nulls: 0 / 389,791 (0.00%)
+----------+----------+
|  earliest|    latest|
+----------+----------+
|2018-01-01|2023-09-03|
+----------+----------+



## 6. Register as Spark SQL tables (optional but useful for querying)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS road_pipeline")

spark.read.format("delta").load(SILVER_SEGMENTS) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("road_pipeline.road_segments_silver")

spark.read.format("delta").load(SILVER_COUNTS) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("road_pipeline.tms_counts_silver")

spark.read.format("delta").load(SILVER_SITES) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("road_pipeline.tms_monitoring_sites_silver")

print("Tables registered in road_pipeline database.")
print("You can now query them with: SELECT * FROM road_pipeline.road_segments_silver")

Tables registered in road_pipeline database.
You can now query them with: SELECT * FROM road_pipeline.road_segments_silver
